# Brain-Act Subject-Specific Whole-Brain Simulations (Rates + TVB BOLD)

This notebook runs subject-specific whole-brain AdEx simulations and saves both temporal-average firing rates and TVB Bold monitor outputs. It computes LZc, PCI, and brain-state analyses in both domains across correlated-noise scenarios.

## Mathematical Setup

Correlated noise mixture per node:

$$\eta_i(t)=\sqrt{1-\alpha}\,\xi_i(t)+\sqrt{\alpha}\,\zeta_i(t),\quad 0\le\alpha\le1$$

BOLD sample target condition:

$$T_{sim} \ge T_{transient} + N_{BOLD}\cdot TR$$

We compare complexity and brain states from both domains (rates and BOLD).

In [ ]:
from __future__ import annotations

from pathlib import Path
from copy import deepcopy
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from time import perf_counter
import multiprocessing as mp
import itertools
import json
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal, mannwhitneyu, spearmanr

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp/matplotlib-cache").resolve()))
os.environ.setdefault("XDG_CACHE_HOME", str(Path("/tmp/xdg-cache").resolve()))
os.environ.setdefault("TVB_USER_HOME", str((PROJECT_ROOT / ".tvb-temp").resolve()))

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tvbtoolkit import WholeBrainConfig, detect_system_specs, list_subjects, run_whole_brain_simulation
from tvbtoolkit.analysis.brain_states import summarize_brain_states
from tvbtoolkit.complexity.measures import lzc_multichannel, pci_casali_like
from tvbtoolkit.datasets.brain_act import load_subject_structural
from tvbtoolkit.workflows.brain_act_dual_domain_parallel import run_dual_domain_job


In [ ]:
# Runtime configuration
DATASET_ROOT = PROJECT_ROOT / "data" / "doc_patients_new_data" / "converted_structural"

FAST_MODE = False
RUN_ALL_SUBJECTS = True
N_SUBJECTS_PER_COHORT = 3
SEEDS = (0,)

# Run profiles:
# - rates_first: fast first-pass on firing rates only (5000 ms simulation)
# - dual_quick: rates + BOLD with ~30 post-transient BOLD points
# - dual_full: rates + BOLD with ~200 post-transient BOLD points
# - dual_2min: rates + BOLD with ~50 post-transient BOLD points (~2 min at TR=2.4 s)
RUN_PROFILE = "dual_2min"  # "rates_first" | "dual_quick" | "dual_full" | "dual_2min"

# Short/simple output roots per profile
RUN_DIR_MAP = {
    "rates_first": "ba_rates",
    "dual_quick": "ba_dual30",
    "dual_full": "ba_dual200",
    "dual_2min": "ba_dual2min",
}
OUT_ROOT = PROJECT_ROOT / "notebooks" / "outputs" / RUN_DIR_MAP[RUN_PROFILE]
FIG_DIR = OUT_ROOT / "figs"
SIM_DIR = OUT_ROOT / "sims"
METRICS_DIR = OUT_ROOT / "res"
for d in (FIG_DIR, SIM_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Parallelisation controls
USE_PARALLEL = True
PARALLEL_BACKEND = "process"   # "process" (recommended) or "thread"
N_WORKERS = max(1, int(round((os.cpu_count() or 8) * 0.85)))
SHOW_PROGRESS = True
PARALLEL_SCOPE = "subject_condition"  # one simulation per (scenario, cohort, subject)

# Keep BLAS/OpenMP from over-threading inside each process
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

index_path = DATASET_ROOT / "index.json"
if not index_path.exists():
    raise FileNotFoundError(f"Dataset index not found: {index_path}")
DATASET_INDEX = json.loads(index_path.read_text())
PREFERRED_COHORT_ORDER = ("coma", "uws", "mcs", "emcs", "control")
COHORTS = tuple(c for c in PREFERRED_COHORT_ORDER if c in set(DATASET_INDEX.get("cohorts", {}).keys()))

# Wes Anderson-inspired palette (high contrast for MCS/UWS separation)
WES_COHORT_BASE = {
    "control": "#5FA7C6",   # aquatic blue
    "emcs": "#8EA65E",      # moss green
    "mcs": "#E1A84A",       # warm mustard
    "uws": "#C2543D",       # brick red
    "coma": "#6B5876",      # muted violet
}
COHORT_COLORS = {c: WES_COHORT_BASE.get(c, "#6C757D") for c in COHORTS}

COHORT_LABELS = {"coma": "COMA", "uws": "UWS", "mcs": "MCS", "emcs": "EMCS", "control": "CONTROL"}

SED_COLORS = {
    "non_sedated": "#5FA7C6",
    "sedated": "#C2543D",
    "unknown": "#9B9286",
}

COMA_SUBGROUP_COLORS = {
    "coma_non_sedated": "#4F86A8",
    "coma_sedated": "#B14A30",
    "coma_unknown": "#7A7269",
}

if RUN_PROFILE == "rates_first":
    ENABLE_BOLD_ANALYSIS = False
    RATE_MONITOR_HZ = 256.0
    BOLD_TARGET_POINTS = 0
    BOLD_TR_S = 2.4
    TRANSIENT_MS = 2000.0
    simulation_length_ms = 10000.0
elif RUN_PROFILE == "dual_quick":
    ENABLE_BOLD_ANALYSIS = True
    RATE_MONITOR_HZ = 128.0
    BOLD_TARGET_POINTS = 30
    BOLD_TR_S = 2.4
    TRANSIENT_MS = 2000.0
    simulation_length_ms = TRANSIENT_MS + BOLD_TARGET_POINTS * (BOLD_TR_S * 1000.0)
elif RUN_PROFILE == "dual_full":
    ENABLE_BOLD_ANALYSIS = True
    RATE_MONITOR_HZ = 128.0
    BOLD_TARGET_POINTS = 200
    BOLD_TR_S = 2.4
    TRANSIENT_MS = 5000.0
    simulation_length_ms = TRANSIENT_MS + BOLD_TARGET_POINTS * (BOLD_TR_S * 1000.0)
elif RUN_PROFILE == "dual_2min":
    ENABLE_BOLD_ANALYSIS = True
    RATE_MONITOR_HZ = 128.0
    BOLD_TARGET_POINTS = 50
    BOLD_TR_S = 2.4
    TRANSIENT_MS = 2000.0
    simulation_length_ms = TRANSIENT_MS + BOLD_TARGET_POINTS * (BOLD_TR_S * 1000.0)
else:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}")

RATE_MONITOR_PERIOD_MS = 1000.0 / RATE_MONITOR_HZ
BOLD_PERIOD_MS = BOLD_TR_S * 1000.0
REQUIRED_SIM_MS = TRANSIENT_MS + BOLD_TARGET_POINTS * BOLD_PERIOD_MS

if FAST_MODE:
    simulation_length_ms = min(simulation_length_ms, 120000.0)

PCI_WINDOW_RATE_MS = 300.0
PCI_WINDOW_BOLD_MS = 30000.0
N_STATES = 5

RUN_LOG_PATH = OUT_ROOT / "run_progress.log"
RUN_LOG_PATH.write_text(f"[{datetime.now().isoformat()}] Brain-Act dual-domain run started\n")

print(detect_system_specs())
print({
    "RUN_PROFILE": RUN_PROFILE,
    "ENABLE_BOLD_ANALYSIS": ENABLE_BOLD_ANALYSIS,
    "FAST_MODE": FAST_MODE,
    "RUN_ALL_SUBJECTS": RUN_ALL_SUBJECTS,
    "cohorts": COHORTS,
    "simulation_length_ms": simulation_length_ms,
    "required_for_target_post_transient_bold_points_ms": REQUIRED_SIM_MS,
    "rate_monitor_hz": RATE_MONITOR_HZ,
    "bold_period_ms": BOLD_PERIOD_MS,
    "bold_target_points": BOLD_TARGET_POINTS,
    "seeds": SEEDS,
    "USE_PARALLEL": USE_PARALLEL,
    "PARALLEL_BACKEND": PARALLEL_BACKEND,
    "N_WORKERS": N_WORKERS,
    "out_root": str(OUT_ROOT),
    "run_log": str(RUN_LOG_PATH),
})

if PARALLEL_SCOPE == "subject_condition" and len(SEEDS) != 1:
    raise ValueError("Parallel scope is subject_condition; set SEEDS to a single value (e.g., (0,)).")


In [ ]:
SCENARIOS = {
    "private_alpha0": {"label": "Private only (alpha=0.00)", "noise_alpha": 0.00, "shared_noise_mode": "none"},
    "global_alpha_low": {"label": "Global shared, low alpha (0.15)", "noise_alpha": 0.15, "shared_noise_mode": "global"},
    "global_alpha_med": {"label": "Global shared, medium alpha (0.40)", "noise_alpha": 0.40, "shared_noise_mode": "global"},
    "global_alpha_high": {"label": "Global shared, high alpha (0.70)", "noise_alpha": 0.70, "shared_noise_mode": "global"},
    "sc_alpha_med": {"label": "SC-shaped shared, medium alpha (0.40)", "noise_alpha": 0.40, "shared_noise_mode": "connectivity"},
}

BASE_PARAMETER_MODEL = {
    "T": 40.0,
    "P_e": [-0.0498, 0.00506, -0.025, 0.0014, -0.00041, 0.0105, -0.036, 0.0074, 0.0012, -0.0407],
    "P_i": [-0.0514, 0.004, -0.0083, 0.0002, -0.0005, 0.0014, -0.0146, 0.0045, 0.0028, -0.0153],
    "E_L_e": -63.0,
    "E_L_i": -65.0,
    "b_e": 5.0,
    "tau_e_e": 5.0,
    "tau_e_i": 5.0,
    "initial_condition": {
        "E": [0.004, 0.004], "I": [0.010, 0.010],
        "C_ee": [0.0, 0.0], "C_ei": [0.0, 0.0], "C_ii": [0.0, 0.0],
        "W_e": [50.0, 50.0], "W_i": [0.0, 0.0], "noise": [0.0, 0.0],
    },
}

def upper_triangle_vector(x):
    arr = np.asarray(x, dtype=float)
    iu = np.triu_indices_from(arr, k=1)
    return np.asarray(arr[iu], dtype=float)

def safe_pearson(a, b):
    xa = np.asarray(a, dtype=float).reshape(-1)
    xb = np.asarray(b, dtype=float).reshape(-1)
    if xa.size != xb.size or xa.size < 3:
        return float("nan")
    if not np.all(np.isfinite(xa)) or not np.all(np.isfinite(xb)):
        return float("nan")
    if np.std(xa) <= 0.0 or np.std(xb) <= 0.0:
        return float("nan")
    return float(np.corrcoef(xa, xb)[0, 1])

def apply_damage_parity(c, l, cohort):
    c = np.asarray(c, dtype=float).copy()
    l = np.asarray(l, dtype=float).copy()
    np.fill_diagonal(c, 0.0)
    np.fill_diagonal(l, 0.0)
    if cohort.lower() in {"mcs", "uws", "emcs"}:
        mismatch = (c == 0.0) & (l != 0.0)
        if np.any(mismatch):
            l[mismatch] = 0.0
    iu = np.triu_indices_from(c, k=1)
    return c, l, float(np.mean(c[iu] == 0.0))

def extract_rate_and_bold(full_output):
    parsed = []
    for t, d in full_output:
        t = np.asarray(t, dtype=float).reshape(-1)
        d = np.asarray(d)
        if d.ndim == 4:
            x = np.asarray(d[:, 0, :, 0], dtype=float)
        elif d.ndim == 2:
            x = np.asarray(d, dtype=float)
        else:
            x = np.asarray(d).reshape(d.shape[0], -1)
        dt = float(np.median(np.diff(t))) if t.size > 1 else np.inf
        parsed.append((dt, t, x))
    parsed.sort(key=lambda z: z[0])
    if len(parsed) < 2:
        raise ValueError("Expected at least two monitor outputs (rates + BOLD).")
    return parsed[0][1], parsed[0][2], parsed[-1][1], parsed[-1][2]

def compute_pci(x, t_ms, post_window_ms):
    x = np.asarray(x, dtype=float)
    t_ms = np.asarray(t_ms, dtype=float)
    if x.shape[0] < 8:
        return float("nan")
    dt_ms = float(np.median(np.diff(t_ms))) if t_ms.size > 1 else 1.0
    stim_idx = x.shape[0] // 2
    max_half = min(stim_idx, x.shape[0] - stim_idx)
    if max_half < 2:
        return float("nan")
    target_bins = max(2, int(round(post_window_ms / max(dt_ms, 1e-9))))
    window_bins = min(max_half, target_bins)
    return float(pci_casali_like(x, stimulation_index=stim_idx, t_analysis_ms=float(window_bins * dt_ms), dt_ms=dt_ms))

def compute_domain_metrics(x, t_ms, sc, n_states, pci_window_ms):
    lzc = float(lzc_multichannel(x))
    pci = compute_pci(x, t_ms, pci_window_ms)
    bs = summarize_brain_states(np.asarray(x, dtype=float), n_states=n_states, random_seed=0, n_init=10)
    occ = np.asarray(bs.occupancy, dtype=float)
    centers = np.asarray(bs.centers, dtype=float)
    sc_vec = upper_triangle_vector(sc)
    sfc = np.asarray([safe_pearson(row, sc_vec) for row in centers], dtype=float)
    order = np.argsort(np.nan_to_num(sfc, nan=np.inf))
    return {
        "lzc": lzc,
        "pci": pci,
        "occupancy_sfc_sorted": occ[order],
        "sfc_sorted": sfc[order],
    }

def holm_correct(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = pvals.size
    order = np.argsort(pvals)
    out = np.empty(m, dtype=float)
    prev = 0.0
    for rank, idx in enumerate(order):
        adj = (m - rank) * pvals[idx]
        adj = max(adj, prev)
        out[idx] = min(adj, 1.0)
        prev = out[idx]
    return out


In [ ]:
cohort_subjects = {}
for cohort in COHORTS:
    subjects = list_subjects(dataset_root=DATASET_ROOT, cohort=cohort)
    sel = subjects if RUN_ALL_SUBJECTS else subjects[:N_SUBJECTS_PER_COHORT]
    cohort_subjects[cohort] = [str(s) for s in sel]

pd.DataFrame([
    {"cohort": c, "n_subjects": len(v), "subjects_preview": ", ".join(v[:6]) + (" ..." if len(v) > 6 else "")}
    for c, v in cohort_subjects.items()
])


## Cohorts Used in This Analysis

This notebook analyses cohorts in this fixed plotting/statistics order:

**COMA → UWS → MCS → EMCS → CONTROL**

- **COMA** (optional): acute coma subgroup when COMA files are present in the converted dataset.
- **UWS**: *Unresponsive Wakefulness Syndrome*.
- **MCS**: *Minimally Conscious State*.
- **EMCS**: *Emergence from Minimally Conscious State*.
- **CONTROL (CNT)**: healthy/control participants.

If COMA data are present, this notebook also reports COMA subgroup splits by sedation (`coma_non_sedated`, `coma_sedated`).

### Why sedation stratification is included

Sedation status is tracked because sedation can alter large-scale dynamics and therefore affect complexity and state-occupancy results.


## Personalised Whole-Brain Model per Subject

For each subject, we build a **subject-specific whole-brain AdEx mean-field model** (Zerlaut first-order) using that subject's:

- structural connectivity matrix `C` (weights), and
- tract-length matrix `L` (delays via conduction speed).

This means each simulation has personalised long-range coupling topology and delay structure.

For patient cohorts (MCS/UWS), lesion/damage information encoded as zeros in structural matrices is preserved; if a mismatch is found where `C_ij = 0` but `L_ij != 0`, tract length is set to zero for parity and consistency.


## Impulsive Response Definition Used for This Notebook

A separate "impulsivity" scalar is **not** computed here. Instead, for PCI computation we define a perturbation-aligned window by using a midpoint pseudo-stimulation index:

1. Let `stimulation_index = T/2` (midpoint in samples of the analysed signal).
2. Use a symmetric analysis window around this midpoint.
3. Set window length from `post_window_ms` and signal sampling interval `dt_ms`.

This gives a consistent perturbation-style segmentation for resting simulations across all subjects/scenarios.


## PCI Computation (Casali-Style in TVBToolkit)

PCI is computed with `pci_casali_like(...)` for both domains (rates and BOLD):

- Input matrix `X` is channels-by-time (internally normalised if needed).
- Baseline and post-perturbation windows are defined around the midpoint pseudo-stimulation index.
- Signals are binarised relative to baseline significance.
- Spatiotemporal binary patterns are sorted and their 2D Lempel-Ziv complexity is computed.
- Complexity is normalised to produce the PCI value.

This is the Casali-style pipeline (not the deprecated ratio proxy).


## Brain-State Estimation and Occupancy

Brain states are computed separately in each domain using `summarize_brain_states(...)`:

1. Build time-resolved state features from the multiregional signal.
2. Cluster into `N_STATES` using k-means.
3. Compute state occupancy probabilities.
4. Compute state-level SC-FC coupling by correlating each state pattern with the subject's structural connectivity upper triangle.
5. Sort states by SC-FC coupling (low → high) and store occupancy in that ordered basis.

This is done for **rates** and **BOLD**, enabling direct domain comparison.


In [ ]:
metric_rows = []
state_rows = []

# Enforce one simulation per (scenario, cohort, subject) job in parallel.
seed0 = int(SEEDS[0])

job_kwargs = [
    {
        "scenario_key": scenario_key,
        "scenario_label": SCENARIOS[scenario_key]["label"],
        "noise_alpha": float(SCENARIOS[scenario_key]["noise_alpha"]),
        "shared_noise_mode": str(SCENARIOS[scenario_key]["shared_noise_mode"]),
        "cohort": cohort,
        "subject_id": subject_id,
        "seed": seed0,
        "dataset_root": str(DATASET_ROOT),
        "sim_dir_root": str(SIM_DIR),
        "simulation_length_ms": float(simulation_length_ms),
        "rate_monitor_period_ms": float(RATE_MONITOR_PERIOD_MS),
        "bold_period_ms": float(BOLD_PERIOD_MS),
        "transient_ms": float(TRANSIENT_MS),
        "n_states": int(N_STATES),
        "pci_window_rate_ms": float(PCI_WINDOW_RATE_MS),
        "pci_window_bold_ms": float(PCI_WINDOW_BOLD_MS),
        "base_parameter_model": deepcopy(BASE_PARAMETER_MODEL),
        "enable_bold": bool(ENABLE_BOLD_ANALYSIS),
    }
    for scenario_key in SCENARIOS.keys()
    for cohort in COHORTS
    for subject_id in cohort_subjects[cohort]
]


def _log_line(msg: str) -> None:
    with RUN_LOG_PATH.open("a") as f:
        f.write(msg + "\n")


_log_line(f"jobs={len(job_kwargs)} backend={PARALLEL_BACKEND} workers={N_WORKERS} scope={PARALLEL_SCOPE}")

if USE_PARALLEL and N_WORKERS > 1:
    if PARALLEL_BACKEND == "process":
        print(f"Dispatching {len(job_kwargs)} jobs on {N_WORKERS} processes")
        executor = ProcessPoolExecutor(max_workers=int(N_WORKERS), mp_context=mp.get_context("spawn"))
    else:
        print(f"Dispatching {len(job_kwargs)} jobs on {N_WORKERS} threads")
        executor = ThreadPoolExecutor(max_workers=int(N_WORKERS))

    with executor as ex:
        submit_ts = {}
        futures = []
        for kw in job_kwargs:
            fut = ex.submit(run_dual_domain_job, **kw)
            submit_ts[fut] = perf_counter()
            futures.append(fut)

        iterator = as_completed(futures)
        if SHOW_PROGRESS and tqdm is not None:
            iterator = tqdm(iterator, total=len(futures), desc="completed jobs")

        n_done = 0
        for fut in iterator:
            n_done += 1
            out = fut.result()
            metric_rows.append(out["metric_row"])
            state_rows.extend(out["state_rows"])

            wall_s = float(perf_counter() - submit_ts[fut])
            runtime_s = float(out.get("runtime_s", float("nan")))
            line = (
                f"[{n_done}/{len(futures)}] scenario={out.get('scenario','')} cohort={out.get('cohort','')} "
                f"subject={out.get('subject_id','')} seed={out.get('seed','')} "
                f"runtime_s={runtime_s:.2f} wall_s={wall_s:.2f}"
            )
            print(line)
            _log_line(line)
else:
    print("Running sequentially")
    iterator = job_kwargs
    if SHOW_PROGRESS and tqdm is not None:
        iterator = tqdm(job_kwargs, desc="scenario/cohort/subject")

    n_done = 0
    for kw in iterator:
        t0 = perf_counter()
        out = run_dual_domain_job(**kw)
        n_done += 1
        metric_rows.append(out["metric_row"])
        state_rows.extend(out["state_rows"])

        wall_s = float(perf_counter() - t0)
        runtime_s = float(out.get("runtime_s", wall_s))
        line = (
            f"[{n_done}/{len(job_kwargs)}] scenario={out.get('scenario','')} cohort={out.get('cohort','')} "
            f"subject={out.get('subject_id','')} seed={out.get('seed','')} "
            f"runtime_s={runtime_s:.2f} wall_s={wall_s:.2f}"
        )
        print(line)
        _log_line(line)

metrics_df = pd.DataFrame(metric_rows)
states_df = pd.DataFrame(state_rows)

# Attach source-file provenance when available (enables COMA subgroup detection).
source_map_path = DATASET_ROOT / "source_subject_map.csv"
if source_map_path.exists():
    src_df = pd.read_csv(source_map_path)
    src_keep = [
        c
        for c in ["subject_id", "source_sc_file", "source_tl_file", "source_subject_index"]
        if c in src_df.columns
    ]
    src_df = src_df[src_keep].drop_duplicates(["subject_id"])
    metrics_df = metrics_df.merge(src_df, on="subject_id", how="left")
    states_df = states_df.merge(src_df, on="subject_id", how="left")
else:
    metrics_df["source_sc_file"] = ""
    states_df["source_sc_file"] = ""


def _derive_coma_subgroup(row: pd.Series) -> str:
    cohort = str(row.get("cohort", "")).strip().lower()
    sed = str(row.get("sedation_group", "")).strip().lower()
    src = str(row.get("source_sc_file", "")).strip().lower()

    has_coma = (cohort == "coma") or ("coma" in src)
    if not has_coma:
        return "non_coma"
    if ("sedated_coma" in src) or (sed == "sedated"):
        return "coma_sedated"
    if ("acute_coma" in src) or (sed == "non_sedated"):
        return "coma_non_sedated"
    return "coma_unknown"


for df in (metrics_df, states_df):
    if not df.empty:
        df["coma_subgroup"] = df.apply(_derive_coma_subgroup, axis=1)

metrics_csv = METRICS_DIR / "dual_domain_metrics.csv"
states_csv = METRICS_DIR / "dual_domain_state_rows.csv"
metrics_df.to_csv(metrics_csv, index=False)
states_df.to_csv(states_csv, index=False)

subject_groups = metrics_df[["cohort", "subject_id", "stage", "sedation_group", "coma_subgroup"]].drop_duplicates(
    ["cohort", "subject_id"]
)
subject_groups_csv = METRICS_DIR / "subject_groups.csv"
subject_groups.to_csv(subject_groups_csv, index=False)

coma_counts = (
    subject_groups[subject_groups["coma_subgroup"].isin(["coma_non_sedated", "coma_sedated", "coma_unknown"])]
    ["coma_subgroup"]
    .value_counts()
    .to_dict()
)
if coma_counts:
    print({"coma_subgroup_subject_counts": coma_counts})

_log_line(f"completed metrics_rows={len(metrics_df)} state_rows={len(states_df)}")
print({
    "metrics_csv": str(metrics_csv),
    "states_csv": str(states_csv),
    "subject_groups_csv": str(subject_groups_csv),
    "run_log": str(RUN_LOG_PATH),
})
print({"metrics_rows": int(metrics_df.shape[0]), "state_rows": int(states_df.shape[0])})
metrics_df.head()


In [ ]:
# Default plotting/statistics pass (annotated, no re-simulation)
# This generates the publication figures with significance stars from saved CSV outputs.

import subprocess

plot_script = PROJECT_ROOT / "scripts" / "plot_stats_from_saved_rates.py"
if not plot_script.exists():
    raise FileNotFoundError(f"Annotated plotting script not found: {plot_script}")

cmd = [sys.executable, str(plot_script), "--root", str(OUT_ROOT)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print({
    "fig_dir": str(FIG_DIR),
    "res_dir": str(METRICS_DIR),
    "fig01": str(FIG_DIR / "fig01_complexity_dual_domain_annotated.svg"),
    "fig02": str(FIG_DIR / "fig02_sfc_vs_occupancy_dual_domain_annotated.svg"),
    "fig03": str(FIG_DIR / "fig03_complexity_by_sedation_facets_annotated.svg"),
})

In [ ]:
# Figure 1 and Figure 2 (annotated)

from IPython.display import SVG, display

fig1 = FIG_DIR / "fig01_complexity_dual_domain_annotated.svg"
fig2 = FIG_DIR / "fig02_sfc_vs_occupancy_dual_domain_annotated.svg"

for p in [fig1, fig2]:
    if not p.exists():
        raise FileNotFoundError(f"Missing figure: {p}")

display(SVG(filename=str(fig1)))
display(SVG(filename=str(fig2)))

In [ ]:
# Figure 3 (sedation comparison in same facet; annotated)

from IPython.display import SVG, display

fig3 = FIG_DIR / "fig03_complexity_by_sedation_facets_annotated.svg"
if not fig3.exists():
    raise FileNotFoundError(f"Missing figure: {fig3}")

display(SVG(filename=str(fig3)))

### Statistical Tests Used (and why)

This notebook uses **pairwise, non-parametric statistics** for complexity metrics and reports regression associations for state coupling:

1. **Figure 1 (cohort differences in LZc/PCI, per scenario)**
- Test: **Mann-Whitney U** for each cohort pair within each scenario.
- Multiple-comparison control: **Holm correction** across all pairwise contrasts for that scenario/metric.
- Why: LZc/PCI distributions are not guaranteed Gaussian, cohort sample sizes are unbalanced, and cohorts are independent.

2. **Figure 3 (sedation split in same scenario)**
- Test A: **Mann-Whitney U** for sedated vs non-sedated **within each cohort** and scenario.
- Test B: **Mann-Whitney U** for sedated vs non-sedated **pooled across cohorts** per scenario.
- Multiple-comparison control: **Holm correction**.
- Why: same non-parametric rationale; this directly answers whether sedation status shifts LZc/PCI under each noise condition.

3. **Figure 2 (SCFC coupling vs occupancy)**
- Association reported: **Spearman rho** (monotonic relation) and linear fit slope p-value (from `linregress`) in the output table.
- Why: Spearman is robust to non-normality/outliers and captures monotonic trends; linear slope is retained as an interpretable effect-size model.

Notes:
- Asterisks (`*`, `**`, `***`) are shown only for contrasts with Holm-corrected significance.
- If no asterisks appear for a panel, no tested contrast passed corrected significance in that run.


In [ ]:
# Significance tables (pairwise only)

pairwise_csv = METRICS_DIR / "stats_pairwise_mannwhitney_holm_with_stars.csv"
sed_pairwise_csv = METRICS_DIR / "stats_pairwise_sedation_within_cohort_holm_with_stars.csv"
sed_scenario_csv = METRICS_DIR / "stats_pairwise_sedation_by_scenario_holm_with_stars.csv"
reg02_csv = METRICS_DIR / "stats_regression_sfc_occupancy_fig02_with_stars.csv"

for pth in [pairwise_csv, sed_pairwise_csv, sed_scenario_csv, reg02_csv]:
    if not pth.exists():
        raise FileNotFoundError(f"Missing stats table: {pth}")

pairwise_df = pd.read_csv(pairwise_csv)
sed_pairwise_df = pd.read_csv(sed_pairwise_csv)
sed_scenario_df = pd.read_csv(sed_scenario_csv)
reg02_df = pd.read_csv(reg02_csv)

print({
    "pairwise_csv": str(pairwise_csv),
    "sed_pairwise_csv": str(sed_pairwise_csv),
    "sed_scenario_csv": str(sed_scenario_csv),
    "reg02_csv": str(reg02_csv),
})

print("\nPairwise cohort contrasts (Holm-corrected; significant only):")
display(pairwise_df[pairwise_df["p_holm"] < 0.05].sort_values(["metric", "scenario", "p_holm"]))

print("\nPairwise sedation contrasts within cohort (Holm-corrected; significant only):")
display(sed_pairwise_df[sed_pairwise_df["p_holm"] < 0.05].sort_values(["metric", "scenario", "cohort", "p_holm"]))

print("\nPairwise sedation contrasts by scenario (Holm-corrected; significant only):")
display(sed_scenario_df[sed_scenario_df["p_holm"] < 0.05].sort_values(["metric", "scenario", "p_holm"]))

print("\nRegression statistics for Figure 2 (kept in table, not overlaid on the figure):")
display(reg02_df)

In [ ]:
# Compact summary table for manuscript text (pairwise + regression)

sig_pairs = pairwise_df[pairwise_df["p_holm"] < 0.05].copy()
sig_sed = sed_pairwise_df[sed_pairwise_df["p_holm"] < 0.05].copy()
sig_sed_scen = sed_scenario_df[sed_scenario_df["p_holm"] < 0.05].copy()

summary = {
    "n_pairwise_cohort_tests": int(len(pairwise_df)),
    "n_pairwise_cohort_significant_holm": int((pairwise_df["p_holm"] < 0.05).sum()) if not pairwise_df.empty else 0,
    "n_pairwise_sedation_within_cohort_tests": int(len(sed_pairwise_df)),
    "n_pairwise_sedation_within_cohort_significant_holm": int((sed_pairwise_df["p_holm"] < 0.05).sum()) if not sed_pairwise_df.empty else 0,
    "n_pairwise_sedation_by_scenario_tests": int(len(sed_scenario_df)),
    "n_pairwise_sedation_by_scenario_significant_holm": int((sed_scenario_df["p_holm"] < 0.05).sum()) if not sed_scenario_df.empty else 0,
    "n_regression_lines_fig02": int(len(reg02_df)),
}
print(summary)

if not sig_pairs.empty:
    sig_pairs["comparison"] = sig_pairs["scenario"] + " | " + sig_pairs["metric"] + " | " + sig_pairs["contrast"]
    display(sig_pairs[["comparison", "p_holm", "stars", "median_a", "median_b"]].sort_values("p_holm"))

if not sig_sed.empty:
    sig_sed["comparison"] = sig_sed["scenario"] + " | " + sig_sed["metric"] + " | " + sig_sed["cohort"]
    display(sig_sed[["comparison", "p_holm", "stars", "median_sedated", "median_non_sedated"]].sort_values("p_holm"))

if not sig_sed_scen.empty:
    sig_sed_scen["comparison"] = sig_sed_scen["scenario"] + " | " + sig_sed_scen["metric"]
    display(sig_sed_scen[["comparison", "p_holm", "stars", "median_sedated", "median_non_sedated"]].sort_values("p_holm"))

In [ ]:
# Output manifest (annotated figures + pairwise stats)

annotated_outputs = {
    "figures": [
        str(FIG_DIR / "fig01_complexity_dual_domain_annotated.svg"),
        str(FIG_DIR / "fig01_complexity_dual_domain_annotated.pdf"),
        str(FIG_DIR / "fig02_sfc_vs_occupancy_dual_domain_annotated.svg"),
        str(FIG_DIR / "fig02_sfc_vs_occupancy_dual_domain_annotated.pdf"),
        str(FIG_DIR / "fig03_complexity_by_sedation_facets_annotated.svg"),
        str(FIG_DIR / "fig03_complexity_by_sedation_facets_annotated.pdf"),
    ],
    "tables": [
        str(METRICS_DIR / "stats_pairwise_mannwhitney_holm_with_stars.csv"),
        str(METRICS_DIR / "stats_pairwise_sedation_within_cohort_holm_with_stars.csv"),
        str(METRICS_DIR / "stats_pairwise_sedation_by_scenario_holm_with_stars.csv"),
        str(METRICS_DIR / "stats_regression_sfc_occupancy_fig02_with_stars.csv"),
    ],
}

for k, v in annotated_outputs.items():
    print(f"{k.upper()}:")
    for pth in v:
        print(" -", pth)

In [ ]:
# Optional: open the output folders quickly (desktop/Jupyter users)

print("Figures:", FIG_DIR)
print("Results:", METRICS_DIR)

# Uncomment on macOS if desired:
# import subprocess
# subprocess.run(["open", str(FIG_DIR)])
# subprocess.run(["open", str(METRICS_DIR)])

## Saved Outputs

- Per-seed arrays: `notebooks/outputs/<run_dir>/sims/.../seed_XXX.npz`
- Metrics tables: `notebooks/outputs/<run_dir>/res/`
- Figures: `notebooks/outputs/<run_dir>/figs/`
- Runtime log: `notebooks/outputs/<run_dir>/run_progress.log`

Key CSV files:
- `dual_domain_metrics.csv`
- `dual_domain_state_rows.csv`
- `subject_groups.csv`
- `stats_kruskal_by_scenario.csv`
- `stats_pairwise_mannwhitney_holm.csv`
- `stats_sedated_vs_non_sedated.csv`
- `assoc_spearman_sfc_occupancy_by_cohort.csv`
- `assoc_spearman_sfc_occupancy_by_sedation.csv`
- `assoc_spearman_sfc_occupancy_by_coma_subgroup.csv`

Additional COMA split figure (if COMA subgroup is present):
- `fig04b_coma_subgroups_sfc_vs_occupancy.svg` / `.pdf`
